In [2]:
import pandas as pd
import sqlite3
import os

In [68]:
tsv_file = "covid_trials.txt"   
db_file = "covid_trials.db"     

# Read the TSV into pandas
df = pd.read_csv(tsv_file, sep="\t")

# Clean column names to be SQL-friendly
df.columns = [c.strip().replace(' ', '_').replace('-', '_') for c in df.columns]

#Convert start_date from MM/DD/YYYY
df['start_date'] = pd.to_datetime(df['start_date'], format='%m/%d/%Y', errors='coerce')
print("Start date conversion done. Sample:")
print(df['start_date'].head())

# Identify numeric columns
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

# Force numeric columns to numeric dtype
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # NaN if cannot convert
# Create SQLite DB and write table
conn = sqlite3.connect(db_file)
df.to_sql("covid_trials", conn, if_exists="replace", index=False)
conn.close()
print(f"Database '{db_file}' created with table 'covid_trials'")

# Helper functions to run SQL queries
def connect_db(db_file=db_file):
    """Connect to the SQLite DB and return connection."""
    return sqlite3.connect(db_file)

def run_query(query, db_file=db_file):
    """Execute SQL query on the given DB and return a DataFrame."""
    conn = connect_db(db_file)
    df_result = pd.read_sql_query(query, conn)
    conn.close()
    return df_result

# Test queries - Count total rows
row_count = run_query("SELECT COUNT(*) AS n_trials FROM covid_trials;")
print("Number of rows:", row_count['n_trials'][0])

# Preview first 5 rows
preview = run_query("SELECT * FROM covid_trials LIMIT 5;")
display(preview)


Start date conversion done. Sample:
0   2020-03-09
1   2020-05-20
2   2020-08-03
3   2020-03-20
4   2020-05-01
Name: start_date, dtype: datetime64[ns]
Database 'covid_trials.db' created with table 'covid_trials'
Number of rows: 9764


,nct_id,title,acronym,other_ids,url,status,why_stopped,hcq,has_dmc,funded_bys,...,minimum_agey,maximum_agey,gender,gender_based,gender_description,healthy_volunteers,population,criteria,study_results,study_documents
0,NCT04448782,Characterization of Reverse Triggering and Oth...,None,RT COVID-19,https://ClinicalTrials.gov/show/NCT04448782,Completed,None,No,No,OTHER,...,18 Years,None,All,None,None,No,Adult patients admitted to the ICU under invas...,Inclusion Criteria: Patients under invasive me...,No,None
1,NCT04448418,The Impact of COVID-19 Outbreak on Trans-popul...,None,TRANSCOVID-19,https://ClinicalTrials.gov/show/NCT04448418,Completed,None,No,No,OTHER,...,18 Years,100 Years,All,None,None,Accepts Healthy Volunteers,transgender population,Inclusion Criteria: transgender subjects age >...,No,None
2,NCT04447690,Prevalence of Mental Health Problems Among Und...,None,FDI UAN1901,https://ClinicalTrials.gov/show/NCT04447690,Completed,None,No,Yes,OTHER|UNKNOWN,...,18 Years,None,All,None,None,Accepts Healthy Volunteers,Undergraduate regular students of every career...,Inclusion Criteria: 18 years old or older Regu...,No,None
3,NCT04447638,Percutaneous Tracheostomy With COVID-19,None,TT17,https://ClinicalTrials.gov/show/NCT04447638,Completed,None,No,No,OTHER,...,18 Years,90 Years,All,None,None,None,Percutaneous tracheostomy performed with aeros...,Inclusion Criteria: who underwent percutaneous...,No,None
4,NCT04445961,Respiratory Mechanics and Gas Exchange in Pati...,COVID-VENT,COVID-VENT,https://ClinicalTrials.gov/show/NCT04445961,Completed,None,No,Yes,OTHER,...,18 Years,90 Years,All,None,None,No,All patients with COVID-19 requiring respirato...,Inclusion Criteria: all patients with COVID-19...,No,None


In [ ]:
# Count total trials

In [7]:
df1 = run_query("""
SELECT COUNT(*) AS total_trials 
FROM covid_trials;
""")
print(df1)

   total_trials
0          9764


In [17]:
df1 = run_query("PRAGMA table_info(covid_trials)")
pd.set_option('display.max_rows', None)
print(df1);

    cid                        name     type  notnull dflt_value  pk
0     0                      nct_id     TEXT        0       None   0
1     1                       title     TEXT        0       None   0
2     2                     acronym     TEXT        0       None   0
3     3                   other_ids     TEXT        0       None   0
4     4                         url     TEXT        0       None   0
5     5                      status     TEXT        0       None   0
6     6                 why_stopped     TEXT        0       None   0
7     7                         hcq     TEXT        0       None   0
8     8                     has_dmc     TEXT        0       None   0
9     9                  funded_bys     TEXT        0       None   0
10   10       sponsor_collaborators     TEXT        0       None   0
11   11                lead_sponsor     TEXT        0       None   0
12   12               collaborators     TEXT        0       None   0
13   13                  study_typ

## General information on Covid-19 trials

In [ ]:
# count the number of trials per each phase

In [45]:
df1 = run_query("""
SELECT phases, COUNT(*) 
FROM covid_trials 
GROUP BY phases
ORDER BY phases
""")                                 
print(df1);

            phases  COUNT(*)
0             None      4029
1    Early Phase 1        72
2   Not Applicable      2689
3          Phase 1       434
4  Phase 1|Phase 2       279
5          Phase 2       972
6  Phase 2|Phase 3       289
7          Phase 3       714
8          Phase 4       286


In [ ]:
# check the distribution of study status among phase 3 trials

In [222]:
df1 = run_query("""
SELECT COUNT (phases) AS Phase3, status
FROM covid_trials
WHERE phases='Phase 3' 
GROUP BY status
ORDER BY status
""")                                 
print(df1);

   Phase3                   status
0      83   Active, not recruiting
1     226                Completed
2       3  Enrolling by invitation
3      52       Not yet recruiting
4     136               Recruiting
5       8                Suspended
6      76               Terminated
7      82           Unknown status
8      48                Withdrawn


In [ ]:
# check the distribution of masking methods among phase3 trials

In [50]:
df1 = run_query("""
SELECT COUNT (phases) AS Phase3, masking
FROM covid_trials
WHERE phases='Phase 3' 
GROUP BY masking
ORDER BY masking
""")                                 
print(df1);

   Phase3            masking
0     131             Double
1     234  None (Open Label)
2     168          Quadruple
3      60             Single
4     121             Triple


In [ ]:
# find how many subjects were enrolled to Covid-19 trials

In [118]:
df1 = run_query("""
SELECT SUM (enrollment)
FROM covid_trials
                   """)
print("Number of subjects enrolled to Covid-19 trials:",df1)

Number of subjects enrolled to Covid-19 trials:    SUM (enrollment)
0       374487794.0


In [ ]:
# find the trial that enrolled the largest number of subjects

In [132]:
df1 = run_query("""
SELECT nct_id, lead_sponsor, interventions, MAX (enrollment) 
FROM covid_trials
                   """)
print("The trial that enrolled the largest number of subjects:/n",df1)

The trial that enrolled the largest number of subjects:/n         nct_id           lead_sponsor  \
0  NCT05697705  AstraZeneca[INDUSTRY]   

                                       interventions  MAX (enrollment)  
0  Other: ChAdOx1 nCOV-19 vaccine (Vaxzeria)|Othe...       155975015.0  


In [ ]:
# find the trial that enrolled the smallest number of subjects

In [134]:
df1 = run_query("""
SELECT nct_id, lead_sponsor, interventions, MIN (enrollment) 
FROM covid_trials
WHERE status="Completed"
                   """)
print("The trial that enrolled the smallest number of subjects:/n",df1)

The trial that enrolled the smallest number of subjects:/n         nct_id                                       lead_sponsor  \
0  NCT05531006  Erol Olcok Corum Training and Research Hospita...   

                       interventions  MIN (enrollment)  
0  Other: retrospective cohort study               1.0  


## Covid19 vaccine trials

In [ ]:
# find the proportion of vaccine trials 

In [218]:
df1 = run_query("""
SELECT ROUND(AVG(LOWER(keywords) LIKE '%accine%'), 2) AS vaccine_proportion
FROM covid_trials;
""")
print((df1))

   vaccine_proportion
0                0.08


In [ ]:
# find the proportion of subjects enrolled to vaccine trials 

In [217]:
df1 = run_query("""
SELECT ROUND(
    1.0 * SUM(CASE WHEN LOWER (keywords) LIKE '%vaccine%' THEN enrollment ELSE 0 END)
    / SUM(enrollment),
    3
) AS vaccine_enrollment_ratio
FROM covid_trials;
""")
print(df1)

   vaccine_enrollment_ratio
0                     0.484


In [ ]:
# find the distribution over time of trials on vaccines

In [216]:
df1 = run_query("""
SELECT strftime('%Y', start_date) AS year,
COUNT(*) AS trial_initiations FROM covid_trials
WHERE LOWER (keywords) LIKE '%vaccine%'  
GROUP BY year
ORDER BY year;                
            """)
print(df1)

   year  trial_initiations
0  2014                  1
1  2018                  1
2  2019                  2
3  2020                 84
4  2021                241
5  2022                134
6  2023                 17
7  2024                  1


In [136]:
# find the distribution of completed vaccine trials over time

In [215]:
df1 = run_query("""
SELECT strftime('%Y', start_date) AS year,
COUNT(*) AS trial_initiations FROM covid_trials
WHERE LOWER (keywords) LIKE '%vaccine%'  
AND status='Completed'
GROUP BY year
ORDER BY year;                
            """)
print(df1)

   year  trial_initiations
0  2019                  1
1  2020                 46
2  2021                 82
3  2022                 19


In [ ]:
# find the companies that sponsored the largest number of vaccine trials

In [214]:
df1 = run_query("""
SELECT lead_sponsor,
COUNT(*) AS trials
FROM covid_trials
WHERE LOWER (keywords) LIKE '%vaccine%'
AND funded_bys = 'INDUSTRY'
GROUP BY lead_sponsor
ORDER BY trials DESC
LIMIT 10;
""")
print(("Number of companies that sponsored the largest number of vaccine trials:",df1))


('Number of companies that sponsored the largest number of vaccine trials:',                                         lead_sponsor  trials
0                        Sinocelltech Ltd.[INDUSTRY]      13
1                              BioNTech SE[INDUSTRY]      12
2                          ModernaTX, Inc.[INDUSTRY]      10
3          Medigen Vaccine Biologics Corp.[INDUSTRY]      10
4                              AstraZeneca[INDUSTRY]       7
5  Anhui Zhifei Longcom Biologic Pharmacy Co., Lt...       5
6                                   Pfizer[INDUSTRY]       4
7                 Novartis Pharmaceuticals[INDUSTRY]       4
8                                 Cinnagen[INDUSTRY]       4
9                                  CureVac[INDUSTRY]       3)


In [ ]:
# find how many of these trials were terminated early

In [213]:
df1 = run_query("""
SELECT lead_sponsor,
COUNT(*) AS trials
FROM covid_trials
WHERE LOWER (keywords) LIKE '%vaccine%'
AND funded_bys = 'INDUSTRY'
AND (status='Suspended' OR status='Withdrawn' OR status='Terminated')
GROUP BY lead_sponsor
ORDER BY trials DESC
LIMIT 10;
""")
print("Number of terminated vaccine trials:",df1)

Number of terminated vaccine trials:                              lead_sponsor  trials
0        Inovio Pharmaceuticals[INDUSTRY]       2
1                       CureVac[INDUSTRY]       2
2                   AstraZeneca[INDUSTRY]       2
3  United Biomedical Inc., Asia[INDUSTRY]       1
4                         Takis[INDUSTRY]       1
5                        Biocad[INDUSTRY]       1
6                         Bayer[INDUSTRY]       1
7   Arcturus Therapeutics, Inc.[INDUSTRY]       1


In [ ]:
# find the number of subjects enrolled to interventional vaccine studies

In [211]:
df1 = run_query("""
SELECT SUM (enrollment)
FROM covid_trials
WHERE LOWER (keywords) LIKE '%vaccine%'
AND study_type='Interventional'
                   """)
print("Number of subjects enrolled to interventional vaccine studies:",df1)

Number of subjects enrolled to interventional vaccine studies:    SUM (enrollment)
0        19807907.0


In [212]:
df1 = run_query("""
SELECT ROUND(AVG(enrollment), 0), phases
FROM covid_trials
WHERE LOWER (keywords) LIKE '%vaccine%'
AND study_type='Interventional'
GROUP BY phases
                   """)
print("The average number of subjects enrolled to interventional vaccine studies per phase:",df1)

The average number of subjects enrolled to interventional vaccine studies per phase:    ROUND(AVG(enrollment), 0)           phases
0                      683.0    Early Phase 1
1                   334172.0   Not Applicable
2                      101.0          Phase 1
3                      361.0  Phase 1|Phase 2
4                      543.0          Phase 2
5                     9675.0  Phase 2|Phase 3
6                     6025.0          Phase 3
7                     2058.0          Phase 4


In [141]:
# find the number of subjects enrolled to observational vaccine studies

In [142]:
df1 = run_query("""
SELECT SUM (enrollment)
FROM covid_trials
WHERE keywords LIKE '%accine%'
AND study_type='Observational'
                   """)
print("Number of subjects enrolled to observational vaccine studies:",df1)

Number of subjects enrolled to observational vaccine studies:    SUM (enrollment)
0       161471943.0


In [ ]:
# count trials focusing on alternative treatments:

In [194]:
df_alter = run_query("""
SELECT
    SUM(LOWER(interventions) LIKE '%yoga%')        AS yoga_count,
    SUM(LOWER(interventions) LIKE '%meditation%')  AS meditation_count,
    SUM(LOWER(interventions) LIKE '%mindfulness%') AS mindfulness_count,
    SUM(LOWER(interventions) LIKE '%tai chi%')     AS tai_chi_count
FROM covid_trials;
""")
print("Counts of trials studing alternative treatments for Covid-19\n",df_alter)

Counts of trials studing alternative treatments for Covid-19
    yoga_count  meditation_count  mindfulness_count  tai_chi_count
0          26                15                 57              2


In [ ]:
# Add data from the CTG-PVAL table, that contains subset of the clinical trials published in the clinicaltrials.gov repository.
# These records contains the p-values of the trials' primary endpoints which indicates whether the trial succeed/failed. 

In [143]:
results = pd.read_csv("CTG_PVAL.csv") 
results.head()

,nct_id,study_result
0,NCT00000378,success
1,NCT00000392,fail
2,NCT00000620,fail
3,NCT00001656,success
4,NCT00001723,success


In [145]:
conn = sqlite3.connect("covid_trials.db") # reopen the connection
results.to_sql(
    name="results",    
    con=conn,            
    if_exists="replace", 
    index=False         
)

22582

In [146]:
df_check = pd.read_sql_query("SELECT * FROM results LIMIT 5;", conn)
print(df_check)

        nct_id study_result
0  NCT00000378      success
1  NCT00000392         fail
2  NCT00000620         fail
3  NCT00001656      success
4  NCT00001723      success


In [206]:
df_results = run_query("""
SELECT
    ct.nct_id,
    ct.study_type,
    ct.keywords,
    ct.enrollment,
    ct.funded_bys,
    ct.interventions,
    ct.title,
    r.study_result
FROM covid_trials AS ct
INNER JOIN results AS r
    ON ct.nct_id = r.nct_id
ORDER BY ct.nct_id
;
""")
print(df_results.head())
print("Number of COVID trials with results:", len(df_results))

        nct_id      study_type  \
0  NCT00042289   Observational   
1  NCT02092467  Interventional   
2  NCT02344290  Interventional   
3  NCT02487251  Interventional   
4  NCT02501811  Interventional   

                                            keywords  enrollment  \
0   Pregnancy|Pharmacokinetics|Treatment Experienced      1578.0   
1  Safety Surveillance|CP690550|Xeljanz|oral trea...      4372.0   
2  HIV|COVID-19|Cardiovascular Disease|Myocardial...      7770.0   
3                Obesity|Family Mealtimes|Prevention       810.0   
4  Heart Failure|Ischemia|Autologous Stem Cells|L...       125.0   

           funded_bys                                      interventions  \
0                 NIH  Drug: atazanavir/cobicistat|Drug: darunavir/ri...   
1            INDUSTRY  Drug: tofacitinib|Drug: tofacitinib|Biological...   
2  NIH|INDUSTRY|OTHER                   Drug: Pitavastatin|Drug: Placebo   
3               OTHER  Behavioral: Mealtime Supports|Behavioral: Usua...   
4     

In [ ]:
# save this JOIN for further analysis

In [207]:
df_results.to_sql("covid_with_results", conn, if_exists="replace", index=False)

312

In [ ]:
# Among the studies that has results, find the distribution of success/failure/unknown:

In [174]:
df_results = run_query("""
SELECT study_result,
       ROUND(1.0 * COUNT(*) / (SELECT COUNT(*) FROM covid_with_results), 3) AS proportion
FROM covid_with_results
GROUP BY study_result
ORDER BY proportion DESC;
""")
print(df_results)


  study_result  proportion
0         fail       0.436
1      success       0.401
2      unknown       0.163


In [175]:
# find the distribution of success among different sponsor types

In [168]:
df_results = run_query("""
SELECT study_result, funded_bys,  
COUNT (*)    
FROM covid_with_results
GROUP BY funded_bys
;
""")
print(df_results)

   study_result                funded_bys  COUNT (*)
0          fail                       FED          5
1       unknown                 FED|OTHER          1
2       success               FED|UNKNOWN          1
3       unknown                  INDUSTRY        141
4          fail              INDUSTRY|FED          1
5       success          INDUSTRY|NETWORK          2
6       success        INDUSTRY|NIH|OTHER          1
7       success            INDUSTRY|OTHER          8
8       success    INDUSTRY|OTHER|UNKNOWN          2
9          fail          INDUSTRY|UNKNOWN          5
10      unknown         NETWORK|NIH|OTHER          1
11      success             NETWORK|OTHER          1
12      success                       NIH          6
13      unknown        NIH|INDUSTRY|OTHER          1
14         fail      NIH|NETWORK|INDUSTRY          1
15         fail        NIH|OTHER|INDUSTRY          1
16         fail                     OTHER         66
17      success                 OTHER_GOV     

In [ ]:
# same analysis, focusing on federal-funded trials:

In [176]:
df_fed = run_query("""
SELECT study_result, funded_bys, COUNT(*) AS n_trials
FROM covid_with_results
WHERE funded_bys LIKE '%FED%'
GROUP BY study_result
ORDER BY n_trials DESC;
""")
print(df_fed)


  study_result   funded_bys  n_trials
0      success  FED|UNKNOWN         6
1         fail          FED         6
2      unknown    FED|OTHER         1


In [ ]:
# same, now for industrial trials:

In [177]:
df_ind = run_query("""
SELECT study_result, funded_bys, COUNT(*) AS n_trials
FROM covid_with_results
WHERE funded_bys LIKE '%INDUSTRY%'
GROUP BY study_result
ORDER BY n_trials DESC;
""")
print(df_ind)


  study_result funded_bys  n_trials
0         fail   INDUSTRY        76
1      success   INDUSTRY        59
2      unknown   INDUSTRY        36


In [ ]:
# check the success of ivermectin trials

In [224]:
df_ind = run_query("""
SELECT study_result, COUNT(*) AS n_trials
FROM covid_with_results
WHERE interventions LIKE '%ivermectin%'
GROUP BY study_result
ORDER BY n_trials DESC;
""")
print("Success rate of ivermectin trials\n",df_ind)

Success rate of ivermectin trials
   study_result  n_trials
0      success         3
1         fail         2
2      unknown         1


In [ ]:
# check the success of hydroxychloroquine trials

In [225]:
df_hcq = run_query("""
SELECT study_result, COUNT(*) AS n_trials
FROM covid_with_results
WHERE LOWER(interventions) LIKE '%hydroxychloroquine%'
GROUP BY study_result
ORDER BY n_trials DESC;
""")
print("Success of hydroxychloroquine trials\n",df_hcq)

Success of hydroxychloroquine trials
   study_result  n_trials
0         fail         4
1      unknown         1
2      success         1


In [ ]:
# check the success of vaccine trials

In [226]:
df_vac = run_query("""
SELECT study_result, COUNT(*) AS n_trials
FROM covid_with_results
WHERE LOWER(title) LIKE '%vaccine%'
GROUP BY study_result
ORDER BY n_trials DESC;
""")
print("Success of vaccine trials\n",df_vac)

Success of vaccine trials
   study_result  n_trials
0      unknown         9
1      success         4
2         fail         3
